In [1]:
! pip install mlflow

In [2]:
! pip install pygam

In [3]:
! pip install "numpy<2.0.0" --force-reinstall

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-dataset-runtime 1.60.0 requires numpy!=1.19.4,<1.24; sys_platform == "win32", but you have numpy 1.26.4 which is incompatible.
azureml-opendatasets 1.60.0 requires pandas<=2.0.0,>=0.21.0, but you have pandas 2.3.3 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
tensorflow-cpu 2.19.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.


In [4]:
! pip install --upgrade --force-reinstall pandas numpy scipy scikit-learn

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-dataset-runtime 1.60.0 requires numpy!=1.19.4,<1.24; sys_platform == "win32", but you have numpy 2.2.6 which is incompatible.
azureml-opendatasets 1.60.0 requires numpy<=2.0.0,>=1.16.0, but you have numpy 2.2.6 which is incompatible.
azureml-opendatasets 1.60.0 requires pandas<=2.0.0,>=0.21.0, but you have pandas 2.3.3 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
tensorflow-cpu 2.19.1 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow-cpu 2.19.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.


In [16]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import skew, kurtosis
import os
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.titlesize'] = 16
sns.set_palette("husl")

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

try:
    from pygam import LinearGAM, s, l
    GAM_AVAILABLE = True
except ImportError:
    GAM_AVAILABLE = False

import mlflow
import mlflow.sklearn

MLFLOW_EXPERIMENT_NAME = "01_Linear_Models_Benchmarking"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

import warnings
warnings.filterwarnings('ignore')


In [10]:
df = pd.read_csv("C:/Users/Asus/montesinho-fire-risk-prediction/data/raw/forestfires.csv")

print("Veri Boyutu:", df.shape)
df.describe()

df.head(10)

Veri Boyutu: (517, 13)


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0
5,8,6,aug,sun,92.3,85.3,488.0,14.7,22.2,29,5.4,0.0,0.0
6,8,6,aug,mon,92.3,88.9,495.6,8.5,24.1,27,3.1,0.0,0.0
7,8,6,aug,mon,91.5,145.4,608.2,10.7,8.0,86,2.2,0.0,0.0
8,8,6,sep,tue,91.0,129.5,692.6,7.0,13.1,63,5.4,0.0,0.0
9,7,5,sep,sat,92.5,88.0,698.6,7.1,22.8,40,4.0,0.0,0.0


In [11]:
y_raw = df['area'].values
y_log = np.log1p(df['area'].values) 

def evaluate_and_log_model(model, X_test, y_test_log, y_test_raw, 
                           model_name, setup_name, feature_names, 
                           scaler_name="StandardScaler", extra_params=None):
    y_pred_log = model.predict(X_test)
    
    log_rmse = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
    log_mae = mean_absolute_error(y_test_log, y_pred_log)
    r2 = r2_score(y_test_log, y_pred_log)
    
    y_pred_raw = np.expm1(y_pred_log)
    y_pred_raw = np.clip(y_pred_raw, a_min=0.0, a_max=None)
    
    ha_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))
    ha_mad = mean_absolute_error(y_test_raw, y_pred_raw) 
    
    zero_mask = (y_test_raw == 0.0)
    zero_day_fp_sum = np.sum(y_pred_raw[zero_mask])
    zero_day_fp_mean = np.mean(y_pred_raw[zero_mask])
    
    with mlflow.start_run(run_name=f"{model_name}_{setup_name}") as run:
        mlflow.log_param("model_family", "Linear_Models")
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("feature_setup", setup_name)
        mlflow.log_param("num_features", len(feature_names))
        mlflow.log_param("features_list", str(feature_names))
        mlflow.log_param("scaler", scaler_name)
        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, v)
        
        mlflow.log_metric("val_log_rmse", log_rmse)
        mlflow.log_metric("val_log_mae", log_mae)
        mlflow.log_metric("val_r2", r2)
        mlflow.log_metric("test_ha_rmse", ha_rmse)
        mlflow.log_metric("test_ha_mad", ha_mad)  
        mlflow.log_metric("zero_day_fp_sum_ha", zero_day_fp_sum)
        mlflow.log_metric("zero_day_fp_mean_ha", zero_day_fp_mean)
        
        try:
            mlflow.sklearn.log_model(model, "model")
        except Exception as e:
            pass
            
    results = {
        "Model": model_name,
        "Setup": setup_name,
        "Num_Features": len(feature_names),
        "R2_Score": r2,
        "Log_RMSE": log_rmse,
        "Hectare_RMSE_ha": ha_rmse,
        "Cortez_MAD_ha": ha_mad,
        "Zero_Day_FP_Sum_ha": zero_day_fp_sum
    }
    
    return results


### Kodun açıklaması
Kuracağımız doğrusal modellerin normal dağılım varsayımını sağlayabilmesi için, aşırı sağa çarpık (`+12.85`) olan hedef değişkenimizi **`np.log1p` ($y = \ln(\text{area} + 1)$)** dönüşümü uyguladık. Ayrıca tahminleri gerçek hektara ($e^{\hat{y}} - 1$) çevirecek bir sistem kurup , Cortez (2007)'nin orijinal **`MAD = 12.71 ha`** hatasıyla adilce kıyaslıyor ve her denememizi tek satır ek kod yazmadan otomatik olarak **MLflow** veritabanına kaydediyoruz.

In [13]:
X = df.drop(columns=['area'])

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.20, random_state=42
)

num_cols = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
cat_cols = ['month', 'day']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f" Eğitim Seti (X_train_scaled) Boyutu : {X_train_scaled.shape[0]} satır, {X_train_scaled.shape[1]} sütun")
print(f" Test Seti   (X_test_scaled)  Boyutu : {X_test_scaled.shape[0]} satır, {X_test_scaled.shape[1]} sütun")
print("-" * 80)
print(" X_train ölçeklendirilmiş ortalama (ilk 3 sütun):")
print(X_train_scaled[num_cols[:3]].mean().round(4).to_dict())
print(" X_train ölçeklendirilmiş standart sapma:")
print(X_train_scaled[num_cols[:3]].std().round(4).to_dict())

 Eğitim Seti (X_train_scaled) Boyutu : 413 satır, 12 sütun
 Test Seti   (X_test_scaled)  Boyutu : 104 satır, 12 sütun
--------------------------------------------------------------------------------
 X_train ölçeklendirilmiş ortalama (ilk 3 sütun):
{'X': 0.0, 'Y': -0.0, 'FFMC': -0.0}
 X_train ölçeklendirilmiş standart sapma:
{'X': 1.0012, 'Y': 1.0012, 'FFMC': 1.0012}


In [14]:
X_train_encoded = pd.get_dummies(X_train_scaled, columns=['month', 'day'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test_scaled, columns=['month', 'day'], drop_first=True)

X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

all_features_list = list(X_train_encoded.columns)

print(all_features_list[:10], "...\n")

['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain'] ...



## DENEME 1 -  HAM VERİNİN 5 DOĞRUSAL MODEL TESTİ

In [17]:
m_cols = list(X_train_encoded.columns)
X_train_M = X_train_encoded.copy()
X_test_M = X_test_encoded.copy()

deneme1_results = []

# 1. OLS (Linear Regression) 
ols_model = LinearRegression().fit(X_train_M, y_train_log)
deneme1_results.append(evaluate_and_log_model(ols_model, X_test_M, y_test_log, y_test_raw, 
                                            "OLS_LinearRegression", "Deneme1_All_Cols", m_cols))

# 2. Ridge Regression
ridge_alphas = np.logspace(-4, 3, 100)
ridge_model = RidgeCV(alphas=ridge_alphas, cv=5, scoring='neg_mean_squared_error').fit(X_train_M, y_train_log)
deneme1_results.append(evaluate_and_log_model(ridge_model, X_test_M, y_test_log, y_test_raw, 
                                            "Ridge_Regression", "Deneme1_All_Cols", m_cols, 
                                            extra_params={"best_alpha": ridge_model.alpha_}))

# 3. Lasso Regressionye
lasso_model = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=5000, n_jobs=-1, random_state=42).fit(X_train_M, y_train_log)
deneme1_results.append(evaluate_and_log_model(lasso_model, X_test_M, y_test_log, y_test_raw, 
                                            "Lasso_Regression", "Deneme1_All_Cols", m_cols, 
                                            extra_params={"best_alpha": lasso_model.alpha_}))

# 4. ElasticNet
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
enet_model = ElasticNetCV(l1_ratio=l1_ratios, alphas=np.logspace(-4, 1, 50), cv=5, max_iter=5000, n_jobs=-1, random_state=42).fit(X_train_M, y_train_log)
deneme1_results.append(evaluate_and_log_model(enet_model, X_test_M, y_test_log, y_test_raw, 
                                            "ElasticNet_Regression", "Deneme1_All_Cols", m_cols, 
                                            extra_params={"best_alpha": enet_model.alpha_, "best_l1_ratio": enet_model.l1_ratio_}))

# 5. GAMs 
if GAM_AVAILABLE:
    try:
        terms = [l(i) if (c.startswith('month_') or c.startswith('day_')) else s(i, n_splines=6) for i, c in enumerate(m_cols)]
        gam_formula = terms[0]
        for t in terms[1:]: gam_formula += t
        gam_model = LinearGAM(gam_formula).gridsearch(X_train_M.values, y_train_log, lam=np.logspace(-2, 2, 5))
        
        class GAMWrapper:
            def __init__(self, obj): self.obj = obj
            def predict(self, X): return self.obj.predict(X.values if hasattr(X, 'values') else X)
            
        deneme1_results.append(evaluate_and_log_model(GAMWrapper(gam_model), X_test_M, y_test_log, y_test_raw, 
                                                    "GAMs_LinearGAM", "Deneme1_All_Cols", m_cols))
    except Exception: pass

df_deneme1 = pd.DataFrame(deneme1_results).sort_values(by="Cortez_MAD_ha", ascending=True).reset_index(drop=True)

display(df_deneme1[["Model", "Setup", "R2_Score", "Log_RMSE", "Hectare_RMSE_ha", "Cortez_MAD_ha", "Zero_Day_FP_Sum_ha"]].round(4))


2026/07/10 01:40:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 01:40:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 01:41:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 01:41:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
  0% (0 of 5) |                          | Elapsed Time: 0:00:00 ETA:  --:--:--
 20% (1 of 5) |#####                     | Elapsed Time: 0:00:00 ETA:   0:00:01
 40% (2 of 5) |##########                | Elapsed Time: 0:00:00 ETA:   0:00:01
 60% (3 of 5) |###############           | Elapsed Time: 0:00:01 ETA:   0:00:00
 80% (4 of 5) |####################      | Elapsed Time: 0:00:01 ETA:   0:00:00
100% (5 of 5) |##########################| Elapsed Time: 0:00:01 Time:  0:00:01
2026/07/10 01:41:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `

,Model,Setup,R2_Score,Log_RMSE,Hectare_RMSE_ha,Cortez_MAD_ha,Zero_Day_FP_Sum_ha
0,GAMs_LinearGAM,Deneme1_All_Cols,0.0106,1.4746,109.8851,19.7980,99.6897
1,Ridge_Regression,Deneme1_All_Cols,0.0058,1.4782,109.9705,19.8073,101.1558
2,Lasso_Regression,Deneme1_All_Cols,-0.0006,1.4830,109.9957,19.8188,102.7652
3,ElasticNet_Regression,Deneme1_All_Cols,-0.0017,1.4838,109.9950,19.8192,102.4571
4,OLS_LinearRegression,Deneme1_All_Cols,-0.0472,1.5171,109.7993,20.1068,145.8549


### Deneme 1 Değerlendirme Raporu

Tüm 12 değişkenin modellere verildiği bu ilk denemedeki sonuçlar:

1. **OLS:** Birbirine bağımlı kuraklık (`FFMC, DMC, DC`) ve hava (`temp, RH, wind`) değişkenlerinin yarattığı **Çoklu Doğrusallık (`Multicollinearity`)**, OLS modelinde katsayı sistemini çökertti. Model **`-0.0472` negatif $R^2$** üretmiş ve, gerçekte hiç yangın çıkmayan sıfır günlerinde toplam **`145.85 hektar` `False Positive` durumu yaşanmış.**
2. **Ridge ve GAMs:** `Ridge` regresyonu, $L_2$ ceza katsayısı ($\alpha$) sayesinde katsayıları düzenleyerek sahte pozitif durumunu **`101.15 hektara`** düşürmüş ve varyansı stabilize etmiştir. Doğrusal olmayan ilişkileri 6 bükümlü eğrilerle (`Splines`) modelleyen **`GAMs (LinearGAM)`**, **`MAD = 19.798 ha`** ve **`99.68 ha`** sahte alarm skoruyla Deneme 1'in en iyi modelidir
3. **Mekan ve Takvim Gürültüsü:** Modellerin `19.79 ha` bandında sıkışması, 14 adet One-Hot takvim sütununun (`month, day`) ve konum koordinatlarının (`X, Y`) doğrusal denklemlerde aşırı seyreklik ve gürültü yarattığını göstermektedir. Bir sonraki aşamada (**Deneme 2**), bu gürültüler temizlenerek sadece saf meteoroloji verisi test edilecektir.